# Dynamic and Hybrid Conditioning for Compositional Image Retrieval

**Deep Learning 2026 — Universita' di Trento**
Arianna Schiavi, Filippo Roseghini, nilo netti

---

Questo notebook e' il documento di riferimento del progetto. Raccoglie la baseline
implementata correttamente, il protocollo di valutazione, e lo stato delle ipotesi testate.
I notebook precedenti sono in `archive/` con un indice che spiega cosa contengono e perche'
non sono piu' la fonte di verita'.

## Sintesi onesta dello stato

| | |
|---|---|
| baseline zero-shot dell'assignment, implementata correttamente | **15.92%** Recall@10 |
| tutte le varianti geometriche costruite finora | nessun guadagno statisticamente dimostrato |
| contributo originale | **da costruire** |

Il progetto ha prodotto finora tre cose solide: un **protocollo di valutazione rigoroso**
(benchmark ufficiale, set di sviluppo a 101 query, bootstrap accoppiato), una **baseline
corretta** — che era sottostimata di oltre tre punti perche' implementata male — e una serie
di **ipotesi escluse con misure oneste**, che dicono dove non vale la pena investire il tempo
rimasto.

Cio' che manca e' il contributo richiesto dalla sez. 3: un modulo di fusione che gestisca
insiemi di condizioni `T+` e `T-`. Le sezioni 5 e 6 spiegano perche' le strade percorse finora
non ci sono arrivate e quali restano aperte.

## 1. Il problema

Dato un'immagine di riferimento `v_ref`, un insieme di condizioni testuali positive `T+` e uno
di negative `T-`, si vuole costruire una query che recuperi immagini che **preservano
l'identita'** di `v_ref` e **soddisfano tutte le condizioni**.

Il criterio di valutazione (sez. 3.1.1) rende preciso "preservare l'identita'": un'immagine e'
ground truth se soddisfa i vincoli e dista al massimo **2 attributi di Hamming** dalla
reference sui restanti. Questa doppia richiesta e' la tensione centrale del problema: spostarsi
per soddisfare le condizioni allontana dall'identita'.

Metriche (sez. 3.1.2): **Recall@K** come hit rate — 1 se almeno un ground truth e' nei primi K
— e Precision@K, a K = 1, 5, 10.

In [1]:
import sys

import numpy as np
import torch
from transformers import CLIPModel, CLIPTokenizer

sys.path.append("..")
from src.benchmark import load_benchmark
from src.devset import evaluate_per_source, macro, bootstrap_paired

MODEL_ID = "openai/clip-vit-base-patch32"      # richiesto dall'assignment, sez. 3.3

def load_split(split):
    d = torch.load(f"../embeddings/celeba_{split}_clip_vitb32.pt", map_location="cpu")
    E = d["embeddings"].float()
    return E / E.norm(dim=-1, keepdim=True), d["labels"], d

E_train, L_train, _ = load_split("train")
E_valid, L_valid, _ = load_split("valid")
E_test,  L_test,  meta = load_split("test")
attr_names = [n for n in meta["attr_names"] if n.strip()]
D = E_test.shape[1]
K_VALUES = [1, 5, 10]

print(f"train {tuple(E_train.shape)} | valid {tuple(E_valid.shape)} | test {tuple(E_test.shape)}")
print(f"attributi: {len(attr_names)}")

train (162770, 512) | valid (19867, 512) | test (19962, 512)
attributi: 40


## 2. Protocollo di valutazione

Due set, con ruoli distinti. Il **benchmark ufficiale** (`celeba_evaluation.json`) e' il metro
finale: 14 entry, 13 query distinte (`-Young` compare due volte), 33052 reference. Il **set di
sviluppo** (`celeba_devset_valid.json`) e' costruito sul validation split con lo stesso
protocollo — verificato esatto contro il file del docente — e serve a scegliere gli
iperparametri senza toccare il test.

### Perche' serve un set di sviluppo a 101 query

La metrica e' una macro-average sulle query. La sua incertezza e' dominata dal **numero di
query**, non dal numero di reference:

| | |
|---|---|
| deviazione standard fra query | 9.30pp |
| errore standard dovuto al campionamento delle query | 2.58pp |
| errore standard dovuto al campionamento delle reference | 0.51pp |

Con 13 query si risolvono solo differenze sopra i ~6.8pp fra metodi lontani, mentre tutti gli
effetti in gioco stanno sotto 1.5pp. **Le conclusioni tratte su 13 query non erano sbagliate
per sfortuna: erano non falsificabili in partenza.** Ogni confronto in questo notebook e'
riportato con la differenza **accoppiata** e il suo intervallo di confidenza, mai con due medie
affiancate.

In [2]:
bench_test = load_benchmark("../celeba_evaluation.json", attr_names)
devset = load_benchmark("../celeba_devset_valid.json", attr_names)

queries = [e["query"] for e in bench_test]
uniq = [i for i, q in enumerate(queries) if q not in queries[:i]]

print(f"benchmark ufficiale : {len(bench_test)} entry, {len(uniq)} query distinte, "
      f"{sum(len(e['sources']) for e in bench_test)} reference")
print(f"set di sviluppo     : {len(devset)} query, "
      f"{sum(len(e['sources']) for e in devset)} reference "
      f"({sum(1 for e in devset if len(e['queried']) > 1)} composte)")

def report(per_source, on_unique=False):
    idx = uniq if on_unique else range(len(per_source[10]))
    return {k: float(np.mean([per_source[k][i].mean() for i in idx])) for k in K_VALUES}

benchmark ufficiale : 14 entry, 13 query distinte, 33052 reference
set di sviluppo     : 101 query, 27206 reference (47 composte)


## 3. Le direzioni di attributo

Ogni condizione testuale diventa una **direzione** nello spazio CLIP, costruita come differenza
fra il prompt che afferma l'attributo e quello che lo nega:

```
d(attr) = normalize( t("... {attr}") - t("... not {attr}") )
```

Fino a poco fa il progetto usava **un solo template**, scelto a maggio e mai verificato. Ma il
paper CLIP dedica una sezione al punto (*Prompt engineering and ensembling*): il modello e'
addestrato su testi naturali completi, e un singolo template arbitrario introduce uno scarto di
distribuzione. La pratica standard e' mediare **80 template** per classe, con un guadagno di
+3.5% su ImageNet.

**Questo appartiene alla baseline, non al contributo.** L'assignment chiede di usare "a
standard, unmodified CLIP model": usare un template solo non e' CLIP standard, e' CLIP usato
male. Qui si adotta un pool di dieci template, fissato a priori e mediato per intero — non si
sceglie il migliore guardando i risultati, che sarebbe la stessa trappola descritta nella
sez. 5.

In [3]:
TEMPLATES = [
    ("a photo of a person who is {}",        "a photo of a person who is not {}"),   # usato finora
    ("a photo of a person with {}",          "a photo of a person without {}"),
    ("a face with {}",                       "a face without {}"),
    ("{}",                                   "not {}"),
    ("a portrait photo of someone with {}",  "a portrait photo of someone with no {}"),
    ("a photo of a {} person",               "a photo of a person that is not {}"),
    ("a cropped photo of a face with {}",    "a cropped photo of a face without {}"),
    ("a close-up photo of a person with {}", "a close-up photo of a person without {}"),
    ("a photo of someone who has {}",        "a photo of someone who has no {}"),
    ("a picture of a face showing {}",       "a picture of a face not showing {}"),
]

clip_model = CLIPModel.from_pretrained(MODEL_ID).eval()
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID)

def encode(prompts):
    with torch.no_grad():
        out = clip_model.get_text_features(**tokenizer(prompts, padding=True,
                                                       return_tensors="pt"))
    return (out.pooler_output if hasattr(out, "pooler_output") else out).float()

d_single = torch.zeros(len(attr_names), D)     # un solo template
d_ens    = torch.zeros(len(attr_names), D)     # ensembling, nella forma usata da CLIP
for i, name in enumerate(attr_names):
    clean = name.replace("_", " ").lower()
    pos = encode([p.format(clean) for p, _ in TEMPLATES])
    neg = encode([n.format(clean) for _, n in TEMPLATES])

    v = pos[0] - neg[0];  d_single[i] = v / v.norm()
    pos = pos / pos.norm(dim=-1, keepdim=True)
    neg = neg / neg.norm(dim=-1, keepdim=True)
    v = pos.mean(0) - neg.mean(0);  d_ens[i] = v / v.norm()

print(f"{len(TEMPLATES)} template x {len(attr_names)} attributi")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

10 template x 40 attributi


### Le direzioni sono davvero migliori?

Controllo diretto sul **train split** (disgiunto da valid e test): quanto bene ciascuna
direzione riconosce il proprio attributo, misurato come AUC.

In [4]:
def auc_per_attr(dirs):
    scores = E_train @ dirs.T
    out = []
    for i in range(len(attr_names)):
        y = (L_train[:, i] == 1).numpy()
        rank = np.argsort(np.argsort(scores[:, i].numpy())) + 1.0
        n_pos, n_neg = int(y.sum()), int((~y).sum())
        out.append(np.nan if min(n_pos, n_neg) < 50
                   else (rank[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))
    return np.array(out)

auc_s, auc_e = auc_per_attr(d_single), auc_per_attr(d_ens)
print(f"AUC media sui 40 attributi:  template singolo {np.nanmean(auc_s):.3f}   "
      f"ensembling {np.nanmean(auc_e):.3f}")
print(f"attributi con direzione INVERTITA (AUC < 0.5): "
      f"{int((auc_s < 0.5).sum())} -> {int((auc_e < 0.5).sum())}")
print(f"\n{'attributo':<20}{'singolo':>10}{'ensemble':>10}")
for i in np.argsort(auc_s)[:5]:
    print(f"{attr_names[i]:<20}{auc_s[i]:>10.3f}{auc_e[i]:>10.3f}")

AUC media sui 40 attributi:  template singolo 0.649   ensembling 0.691
attributi con direzione INVERTITA (AUC < 0.5): 9 -> 7

attributo              singolo  ensemble
Sideburns                0.316     0.495
Rosy_Cheeks              0.414     0.488
5_o_Clock_Shadow         0.418     0.447
Wearing_Necklace         0.418     0.481
Oval_Face                0.440     0.497


Con un solo template **nove attributi su quaranta hanno la direzione invertita** rispetto alla
label: su quelle condizioni il conditioning non poteva funzionare, qualunque fosse il modo di
combinarle. E' un difetto della rappresentazione, non del metodo di fusione — ed e' il motivo
per cui nessuna delle varianti geometriche provate poteva recuperare molto.

## 4. La baseline

La sez. 3.2 punto 3 la definisce senza ambiguita': CLIP standard e aritmetica latente ingenua,

```
v_target ~= v_ref + t_glasses - t_red_hair
```

Due dettagli su cui il progetto ha sbagliato a lungo, entrambi corretti qui:

- la formula **non normalizza** la somma delle direzioni. I notebook precedenti la
  normalizzavano, ottenendo una baseline piu' debole di quella richiesta;
- le direzioni vanno costruite **usando CLIP come si deve**, cioe' con l'ensembling.

In [5]:
def build_query(dirs, alpha=1.0, unit=False):
    """q = normalize(ref + alpha * d). Con `unit=False` d e' la somma grezza delle
    direzioni, come nella formula dell'assignment."""
    cached = {}
    def build(ref, entry):
        key = entry["query"]
        if key not in cached:
            d = torch.zeros(D)
            for i in entry["pos"]: d = d + dirs[i]
            for j in entry["neg"]: d = d - dirs[j]
            cached[key] = d / d.norm().clamp(min=1e-8) if unit else d
        q = ref + alpha * cached[key]
        return q / q.norm(dim=-1, keepdim=True)
    return build

def only_ref(ref, entry):
    return ref                                  # nessuna condizione: il lower bound

METHODS = {
    "solo reference (lower bound)":        only_ref,
    "baseline, template singolo":          build_query(d_single),
    "baseline, con ensembling":            build_query(d_ens),
}
ps_test = {nm: evaluate_per_source(fn, bench_test, E_test, K_VALUES)
           for nm, fn in METHODS.items()}
ps_dev  = {nm: evaluate_per_source(fn, devset, E_valid, K_VALUES)
           for nm, fn in METHODS.items()}

print("BENCHMARK UFFICIALE (13 query distinte)")
print(f"{'metodo':<32}" + "".join(f"{f'R@{k}':>9}" for k in K_VALUES))
for nm in METHODS:
    r = report(ps_test[nm], on_unique=True)
    print(f"{nm:<32}" + "".join(f"{r[k]*100:>8.2f}%" for k in K_VALUES))

print("\nSET DI SVILUPPO (101 query)")
print(f"{'metodo':<32}" + "".join(f"{f'R@{k}':>9}" for k in K_VALUES))
for nm in METHODS:
    r = report(ps_dev[nm])
    print(f"{nm:<32}" + "".join(f"{r[k]*100:>8.2f}%" for k in K_VALUES))

BENCHMARK UFFICIALE (13 query distinte)
metodo                                R@1      R@5     R@10
solo reference (lower bound)        1.73%    5.29%    7.62%
baseline, template singolo          3.31%    8.87%   13.39%
baseline, con ensembling            3.90%   10.99%   15.92%

SET DI SVILUPPO (101 query)
metodo                                R@1      R@5     R@10
solo reference (lower bound)        3.23%   10.14%   14.70%
baseline, template singolo          3.97%   12.46%   18.49%
baseline, con ensembling            4.22%   13.34%   19.91%


In [6]:
def compare(ps, a, b, label):
    print(f"{label}  ({a}  vs  {b})")
    for k in K_VALUES:
        r = bootstrap_paired(ps[a][k], ps[b][k], n_boot=20000, seed=0)
        star = "  significativo" if r["p"] < 0.05 else ""
        print(f"    R@{k:<4}{r['delta']*100:+6.2f}pp  "
              f"IC [{r['lo']*100:+.2f},{r['hi']*100:+.2f}]  p={r['p']:.4f}{star}")
    print()

compare(ps_test, "baseline, con ensembling", "baseline, template singolo",
        "BENCHMARK — quanto vale usare CLIP come si deve")
compare(ps_dev, "baseline, con ensembling", "baseline, template singolo",
        "SVILUPPO — quanto vale usare CLIP come si deve")
compare(ps_test, "baseline, con ensembling", "solo reference (lower bound)",
        "BENCHMARK — quanto vale il conditioning in se'")

BENCHMARK — quanto vale usare CLIP come si deve  (baseline, con ensembling  vs  baseline, template singolo)
    R@1    +0.56pp  IC [-0.48,+1.52]  p=0.2363
    R@5    +2.03pp  IC [+0.95,+3.31]  p=0.0000  significativo
    R@10   +2.41pp  IC [+1.06,+4.06]  p=0.0000  significativo

SVILUPPO — quanto vale usare CLIP come si deve  (baseline, con ensembling  vs  baseline, template singolo)


    R@1    +0.24pp  IC [-0.06,+0.54]  p=0.1128


    R@5    +0.88pp  IC [+0.29,+1.49]  p=0.0018  significativo


    R@10   +1.42pp  IC [+0.75,+2.14]  p=0.0000  significativo

BENCHMARK — quanto vale il conditioning in se'  (baseline, con ensembling  vs  solo reference (lower bound))
    R@1    +2.02pp  IC [+0.45,+3.90]  p=0.0122  significativo
    R@5    +5.37pp  IC [+2.15,+8.86]  p=0.0017  significativo
    R@10   +7.85pp  IC [+3.61,+12.56]  p=0.0005  significativo



### Lettura

L'ensembling vale **oltre due punti** di Recall@10 sul benchmark, con p < 0.001, e il guadagno
si conferma sul set di sviluppo. Non e' un contributo: e' la baseline implementata come
l'assignment la definisce.

Il secondo confronto da' la misura del problema. Cercare **senza alcuna condizione**, con la
sola immagine di riferimento, recupera gia' una quota rilevante del ground truth: e' l'effetto
del vincolo di Hamming, che premia la conservazione dell'identita' — cosa che CLIP fa da solo.
Tutto il conditioning vale la differenza fra le due righe, ed e' dentro quel margine che ogni
metodo deve trovare spazio.

## 5. Ipotesi testate e scartate

Questa sezione e' il risultato metodologico principale del progetto finora. Ogni riga e' stata
misurata con **oracoli onesti** — si sceglie il valore ottimale su meta' del ground truth e lo
si valuta sull'altra meta' — oppure con il bootstrap accoppiato. I dettagli sono nei notebook
in `archive/`.

| ipotesi | margine apparente | margine reale | dove |
|---|---|---|---|
| calibrare la lunghezza del passo | +3.4pp | non significativo, p=0.109 | `archive/07`, rettificato |
| passo adattato a ogni reference | +6.2pp | **nullo** | `archive/08` |
| ortogonalizzare rispetto agli altri attributi | +3.7pp | non significativo con direzioni corrette | `archive/05b`, `archive/07` |
| pesare meno le condizioni negative | +0.48pp | rumore, p=0.36 | `archive/08`, `archive/09` |
| pesi per condizione, per reference | +3.8pp | +0.1pp, non significativo | `archive/10` |
| pesi per condizione, per query | +1.4pp su 6 query | +0.3pp su 47 query, non significativo | `archive/10` |
| modulo Phi appreso (MLP + InfoNCE) | — | **peggiora** il metodo statico | `archive/08` |
| normalizzare la somma e calibrare il passo | sembra aiutare su 13 query | -0.20pp su 101 query, non significativo | questo notebook, sez. 6 |

**Lo schema si ripete.** Ogni volta il margine apparente veniva da un massimo preso su N
alternative, valutato sugli stessi dati usati per sceglierlo. Con un hold-out onesto o un
campione piu' grande, spariva. E' la ragione per cui il set di sviluppo a 101 query e il
bootstrap accoppiato non sono un dettaglio metodologico ma il principale strumento di lavoro:
senza, il progetto ha inseguito per settimane differenze che non esistevano.

**Il modulo appreso merita una nota**, perche' e' l'unico caso in cui il fallimento e'
istruttivo e non solo statistico. Un MLP che predice passo e pesi da feature geometriche,
addestrato con InfoNCE su query mai viste, **peggiora** il metodo statico. La causa non e'
l'architettura: e' che la loss veniva calcolata su poche decine di distrattori, e su quel
campione non e' un surrogato del ranking globale. Corretta la funzione obiettivo (denominatore
sull'intero corpus), il modulo converge da solo a un passo quasi costante — cioe' scopre che
non c'e' nulla da adattare. Gli oracoli onesti confermano indipendentemente: non c'e' margine.

## 6. Il passo e l'ortogonalizzazione, con le direzioni corrette

Le due manopole su cui il progetto ha lavorato di piu' vanno rimisurate ora che le direzioni
sono costruite bene: quel che sembravano guadagni potrebbe essere stato il recupero parziale di
direzioni difettose.

In [7]:
ALPHAS = [0.5, 1.0, 1.5, 2.0, 2.5]

def with_ortho(dirs, alpha):
    cached = {}
    def build(ref, entry):
        key = entry["query"]
        if key not in cached:
            d = torch.zeros(D)
            for i in entry["pos"]: d = d + dirs[i]
            for j in entry["neg"]: d = d - dirs[j]
            C = dirs[entry["non_queried"]]                    # gli attributi non interrogati
            _, S, Vh = torch.linalg.svd(C, full_matrices=False)
            B = Vh[: int((S > 1e-6).sum())]
            d = d - B.T @ (B @ d)                             # via la componente comune
            cached[key] = d / d.norm().clamp(min=1e-8)
        q = ref + alpha * cached[key]
        return q / q.norm(dim=-1, keepdim=True)
    return build

sweep = {}
for nm, fn in [("passo calibrato", lambda a: build_query(d_ens, a, unit=True)),
               ("passo + ortogonalizzazione", lambda a: with_ortho(d_ens, a))]:
    sweep[nm] = {a: evaluate_per_source(fn(a), devset, E_valid, K_VALUES) for a in ALPHAS}
    print(f"{nm:<28}" + "".join(f"{macro(sweep[nm][a][10])*100:>7.2f}%" for a in ALPHAS))
print(f"{'(alpha)':<28}" + "".join(f"{a:>8}" for a in ALPHAS))

best = {nm: max(ALPHAS, key=lambda a: macro(sweep[nm][a][10])) for nm in sweep}
ps_dev["passo calibrato"] = sweep["passo calibrato"][best["passo calibrato"]]
ps_dev["passo + ortogonalizzazione"] = sweep["passo + ortogonalizzazione"][best["passo + ortogonalizzazione"]]
print("\npasso migliore:", best)

passo calibrato               17.19%  19.09%  19.71%  18.75%  17.37%


passo + ortogonalizzazione    17.36%  19.32%  19.55%  18.27%  16.65%
(alpha)                          0.5     1.0     1.5     2.0     2.5

passo migliore: {'passo calibrato': 1.5, 'passo + ortogonalizzazione': 1.5}


In [8]:
compare(ps_dev, "passo calibrato", "baseline, con ensembling",
        "SVILUPPO — il passo calibrato aggiunge qualcosa alla baseline corretta?")
compare(ps_dev, "passo + ortogonalizzazione", "passo calibrato",
        "SVILUPPO — l'ortogonalizzazione aggiunge qualcosa?")

SVILUPPO — il passo calibrato aggiunge qualcosa alla baseline corretta?  (passo calibrato  vs  baseline, con ensembling)


    R@1    -0.05pp  IC [-0.37,+0.27]  p=0.7515


    R@5    -0.43pp  IC [-1.05,+0.21]  p=0.1814


    R@10   -0.20pp  IC [-0.99,+0.63]  p=0.6117

SVILUPPO — l'ortogonalizzazione aggiunge qualcosa?  (passo + ortogonalizzazione  vs  passo calibrato)


    R@1    -0.09pp  IC [-0.50,+0.32]  p=0.6786


    R@5    +0.17pp  IC [-0.57,+0.91]  p=0.6490


    R@10   -0.16pp  IC [-1.04,+0.68]  p=0.7079



### Lettura

**Nessuna delle due aggiunge un guadagno dimostrabile** una volta che le direzioni sono
costruite correttamente. Sul benchmark ufficiale sembrano aiutare, ma con 13 query quella
differenza non e' distinguibile dal rumore; sul set a 101 query, dove la differenza *sarebbe*
risolvibile, non c'e'.

E' la conclusione piu' scomoda del progetto: **il lavoro geometrico svolto da maggio non
produce un miglioramento misurabile sopra la baseline implementata bene.** In buona parte
stava compensando direzioni difettose — e una volta corrette quelle, non resta.

## 7. Dove sta il contributo, e perche' non e' ancora qui

L'assignment (sez. 3) chiede un modulo di fusione `Phi` che accetti insiemi `T+` e `T-` e
decida **dinamicamente** come pesarli, superando la concatenazione ingenua pre-SVD di CLAY.
Quello che il progetto ha finora non e' quel modulo: e' la baseline, misurata bene.

**Cio' che gli oracoli escludono.** Adattare il passo alla reference: margine nullo. Pesare le
condizioni una per una: frazione di punto, non significativa. Queste due strade sono chiuse, e
la loro chiusura e' un risultato — dice dove non spendere il tempo rimasto.

**Cio' che resta aperto**, in ordine di quanto e' sostenuto dai dati raccolti:

1. **Le direzioni deboli.** Anche con l'ensembling restano attributi con la direzione invertita
   rispetto alla label. Su quelli nessun metodo di fusione puo' funzionare, perche' il segnale
   non c'e'. E' il difetto piu' grande documentato, e agisce a monte di qualunque `Phi`.
   *Ostacolo noto*: il set di sviluppo a 101 query **esclude** questi attributi per costruzione
   — servivano query dove i metodi si distinguessero — quindi su questo fronte e' cieco. Serve
   un secondo set con il pool completo, `build_stratified_devset` con tutti gli attributi.
2. **La fusione congiunta delle condizioni.** Tutto il lavoro finora costruisce una direzione
   per attributo e poi le combina. Nessuno ha provato a costruire la direzione **congiunta**
   di `T+` e `T-` — per esempio generando prompt che descrivono la combinazione, come CLAY fa
   con un LLM, invece di prompt separati da sommare. E' l'unica strada rimasta che sia insieme
   specifica del multi-condizione e non ancora esclusa da una misura.
3. **Il vincolo di identita' trattato esplicitamente.** Il criterio di Hamming premia la
   conservazione, e la sola reference recupera gia' molto. Un metodo che modelli il vincolo
   invece di sperare che la vicinanza in CLIP lo garantisca lavorerebbe sulla parte grande del
   problema.

**Vincolo metodologico da qui in avanti**: ogni confronto va riportato con la differenza
accoppiata e il suo intervallo, sul set a 101 query, mai con due medie affiancate su 13. Tre
conclusioni del progetto sono state rovesciate per non averlo fatto.

## Riferimenti

- Radford et al. (2021), *Learning Transferable Visual Models From Natural Language
  Supervision* (CLIP) — sez. *Prompt engineering and ensembling*.
- Berasi et al. (2025), *Not only text: Exploring compositionality of visual representations in
  vision-language models* (GDE), CVPR.
- Lim et al. (2026), *CLAY: Conditional visual similarity modulation in vision-language
  embedding space*, CVPR.
- Liu et al. (2015), *Deep Learning Face Attributes in the Wild* (CelebA), ICCV.

## Struttura del repository

```
src/benchmark.py    valutazione sul benchmark ufficiale; ricostruzione del protocollo
src/devset.py       set di sviluppo ampliato, bootstrap gerarchico e accoppiato
src/extract_embeddings.py   estrazione delle feature visive (CLIP ViT-B/32 da HuggingFace)
celeba_evaluation.json      benchmark ufficiale del docente
celeba_devset_valid.json    set di sviluppo, 101 query sul validation split
notebooks/archive/          i notebook precedenti, con un indice di cosa contengono
docs/                       i report di sessione, in ordine cronologico
```

**Nota tecnica pendente.** Gli embedding in `embeddings/*.pt` sono stati estratti il 13/05 con
la libreria `clip` di OpenAI anziche' con `transformers`. I pesi sono gli stessi e un controllo
di sanita' (AUC zero-shot media 0.786) conferma che sono validi, ma per conformita' alla
sez. 3.3 vanno ri-estratti prima della consegna, in un unico passaggio su tutti e tre gli split.